## Making a football fair team splitter

In [412]:
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import MinMaxScaler, LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from tabulate import tabulate #for creating tables

In [413]:
df = pd.read_csv('zoki_liga_formatted.csv')

In [414]:
df

,index,name,position,secondary_position,matches,goals,assists,goal_contribution,gc_per_match,hat-trick,...,interceptions_overall_ability,fouls_commiting,stamina_rating,positions_played,leadership_measure,passing,physical,gk_ability,skill,positioning_sense
0,1,Daco,D,GK,36,5,11,16,0.44,0,...,5.50,4.25,5.00,5.00,3.25,4.00,5.75,9.50,3.00,4.50
1,2,Leo,D,GK,34,40,19,59,1.74,3,...,8.50,5.00,8.50,9.00,7.50,8.25,9.00,7.25,7.00,8.00
2,3,Simikj,M,NaN,34,10,24,34,1.00,0,...,5.25,2.75,4.25,4.00,2.75,7.50,2.00,1.25,7.50,6.50
3,4,Stef,M,GK,32,22,12,34,1.06,2,...,5.00,3.00,5.50,4.75,5.75,4.25,4.50,5.50,3.50,4.75
4,5,Neno,GK,NaN,29,0,6,6,0.21,0,...,3.00,8.50,5.00,2.75,1.00,3.25,5.75,8.00,2.50,2.25
5,6,Prem,A,NaN,27,38,19,57,2.11,7,...,6.75,2.75,6.25,5.75,6.50,7.75,6.75,3.25,8.50,8.00
6,7,Borjan,M,GK,27,31,19,50,1.85,4,...,4.50,4.25,5.50,4.50,4.75,5.25,8.25,4.75,5.50,7.00
7,8,Ancho,D,NaN,27,2,0,2,0.07,0,...,4.00,5.75,5.25,2.00,2.75,2.00,3.00,1.50,2.00,3.75
8,9,Hito,A,NaN,26,57,22,79,3.04,10,...,8.25,9.00,8.75,6.00,7.00,7.50,8.00,3.00,8.00,8.25
9,10,Hristijan,D,NaN,25,14,21,35,1.40,0,...,6.50,3.50,4.00,4.75,4.75,6.25,8.00,3.00,5.50,7.00


In [415]:
df = df.drop(columns=['index','goal_contribution','owngoals','freekicks','penalty_saves','penalty_stats','w','losses','nominations','gotm'])

In [416]:
df.head(10)

,name,position,secondary_position,matches,goals,assists,gc_per_match,hat-trick,hat-trick-assists,w_percentage,...,interceptions_overall_ability,fouls_commiting,stamina_rating,positions_played,leadership_measure,passing,physical,gk_ability,skill,positioning_sense
0,Daco,D,GK,36,5,11,0.44,0,0,0.36,...,5.50,4.25,5.00,5.00,3.25,4.00,5.75,9.50,3.0,4.50
1,Leo,D,GK,34,40,19,1.74,3,3,0.35,...,8.50,5.00,8.50,9.00,7.50,8.25,9.00,7.25,7.0,8.00
2,Simikj,M,NaN,34,10,24,1.00,0,1,0.41,...,5.25,2.75,4.25,4.00,2.75,7.50,2.00,1.25,7.5,6.50
3,Stef,M,GK,32,22,12,1.06,2,0,0.44,...,5.00,3.00,5.50,4.75,5.75,4.25,4.50,5.50,3.5,4.75
4,Neno,GK,NaN,29,0,6,0.21,0,0,0.38,...,3.00,8.50,5.00,2.75,1.00,3.25,5.75,8.00,2.5,2.25
5,Prem,A,NaN,27,38,19,2.11,7,0,0.41,...,6.75,2.75,6.25,5.75,6.50,7.75,6.75,3.25,8.5,8.00
6,Borjan,M,GK,27,31,19,1.85,4,1,0.63,...,4.50,4.25,5.50,4.50,4.75,5.25,8.25,4.75,5.5,7.00
7,Ancho,D,NaN,27,2,0,0.07,0,0,0.37,...,4.00,5.75,5.25,2.00,2.75,2.00,3.00,1.50,2.0,3.75
8,Hito,A,NaN,26,57,22,3.04,10,2,0.46,...,8.25,9.00,8.75,6.00,7.00,7.50,8.00,3.00,8.0,8.25
9,Hristijan,D,NaN,25,14,21,1.40,0,1,0.52,...,6.50,3.50,4.00,4.75,4.75,6.25,8.00,3.00,5.5,7.00


In [417]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44 entries, 0 to 43
Data columns (total 23 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   name                           44 non-null     object 
 1   position                       44 non-null     object 
 2   secondary_position             5 non-null      object 
 3   matches                        44 non-null     int64  
 4   goals                          44 non-null     int64  
 5   assists                        44 non-null     int64  
 6   gc_per_match                   44 non-null     float64
 7   hat-trick                      44 non-null     int64  
 8   hat-trick-assists              44 non-null     int64  
 9   w_percentage                   44 non-null     float64
 10  loss_percentage                44 non-null     float64
 11  motm                           44 non-null     int64  
 12  potm                           44 non-null     int64

In [418]:
df.columns.tolist()

['name',
 'position',
 'secondary_position',
 'matches',
 'goals',
 'assists',
 'gc_per_match',
 'hat-trick',
 'hat-trick-assists',
 'w_percentage',
 'loss_percentage',
 'motm',
 'potm',
 'interceptions_overall_ability',
 'fouls_commiting',
 'stamina_rating',
 'positions_played',
 'leadership_measure',
 'passing',
 'physical',
 'gk_ability',
 'skill',
 'positioning_sense']

## Attackers
The formula and the stats are set depending on the performance and on the attributes I think football is requiring for an ATTACKER to be complete.
I add more bias to the goals assists, to the physicality, to the skill and to passing.

#### I will try to use the Z-Score now to calculate for the attackers, their ratings for the sheet so i can make the split in a more better way, using the important stats for the position I need.

In [419]:
attackers = df[df['position'] == 'A'].copy()

In [420]:
weights_attack = {
    'goals': 0.29,
    'stamina_rating': 0.13,
    'gc_per_match': 0.12,
    'physical': 0.09,
    'assists': 0.08,
    'positioning_sense': 0.08,
    'positions_played': 0.08,
    'passing': 0.07,
    'leadership_measure': 0.07,
    'skill': 0.06,
    'fouls_commiting': -0.07
}

In [421]:
attacker_features = list(weights_attack.keys())
X_attack = attackers[attacker_features].apply(pd.to_numeric, errors="coerce")

In [422]:
mu_attack = X_attack.mean()
sigma_attack = X_attack.std(ddof=0)
sigma_attack = sigma_attack.replace(0, 1e-9)
Z_attack = (X_attack - mu_attack) / sigma_attack
w_attack = pd.Series(weights_attack) #weihts
attacker_score = (Z_attack * w_attack).sum(axis=1)
attackers["attacker_score"] = attacker_score
attackers["rating"] = 50 + 10 * attacker_score

In [423]:
attackers_matches = df[df['position'] == 'A']['matches'].copy()

In [424]:
min_matches_attack = attackers_matches.min()
max_matches_attack = attackers_matches.max()
experience_factor_a = 0.85 + 0.15 * (attackers_matches - min_matches_attack) / (max_matches_attack - min_matches_attack)
attackers['rating'] = attackers['rating'] * experience_factor_a

In [425]:
attackers = attackers.sort_values(by='rating', ascending=False)

In [426]:
attackers[['name','rating']].round(2)

,name,rating
8,Hito,66.30
5,Prem,58.35
11,Dzhoker,47.97
36,Tomas,42.85
34,Gjozo,42.03
33,Bodan,41.83
17,Lazo,41.56
35,Gashtar,39.77
37,Jan,38.77
26,Nikec,32.28


In [427]:
avg_attacker_rating = round((attackers['rating'].sum())/attackers['rating'].size, 3)
print(f"Average Rating of Attacker is {avg_attacker_rating}")

Average Rating of Attacker is 45.171


## Midfield

In [428]:
midfielders = df[df['position'] == 'M'].copy()

In [429]:
midfielders

,name,position,secondary_position,matches,goals,assists,gc_per_match,hat-trick,hat-trick-assists,w_percentage,...,interceptions_overall_ability,fouls_commiting,stamina_rating,positions_played,leadership_measure,passing,physical,gk_ability,skill,positioning_sense
2,Simikj,M,NaN,34,10,24,1.00,0,1,0.41,...,5.25,2.75,4.25,4.00,2.75,7.50,2.00,1.25,7.50,6.50
3,Stef,M,GK,32,22,12,1.06,2,0,0.44,...,5.00,3.00,5.50,4.75,5.75,4.25,4.50,5.50,3.50,4.75
6,Borjan,M,GK,27,31,19,1.85,4,1,0.63,...,4.50,4.25,5.50,4.50,4.75,5.25,8.25,4.75,5.50,7.00
12,Grekos,M,NaN,22,7,5,0.55,0,0,0.23,...,4.50,3.50,6.00,4.50,2.00,4.50,3.25,1.75,3.75,5.25
14,Mario,M,NaN,11,13,8,1.91,0,0,0.55,...,8.50,7.00,8.25,5.75,7.25,8.25,7.25,2.75,7.50,8.50
16,Dare,M,NaN,10,11,9,2.00,1,1,0.30,...,6.75,2.00,6.00,6.00,3.25,8.25,4.25,2.00,7.25,7.75
18,Martin,M,NaN,9,9,7,1.78,2,0,0.33,...,8.50,3.25,8.25,8.25,7.50,9.00,8.75,3.50,9.25,9.00
20,Nargo,M,NaN,4,4,3,1.75,0,0,0.25,...,6.00,6.00,6.00,4.00,4.00,7.00,4.00,1.00,8.00,6.00
22,Filip,M,NaN,4,1,0,0.25,0,0,0.50,...,4.00,2.33,3.67,4.33,2.33,4.00,3.00,3.67,3.67,4.67
23,Vule,M,NaN,3,8,6,4.67,1,1,1.00,...,8.50,2.75,9.25,7.75,8.25,8.75,9.75,4.50,9.50,9.25


In [430]:
weights_midfielders = {
    'passing': 0.15,
    'assists': 0.15,
    'skill': 0.13,
    'positioning_sense': 0.13,
    'stamina_rating': 0.13,
    'interceptions_overall_ability': 0.12,
    'goals': 0.1,
    'leadership_measure': 0.09,
    'physical': 0.07,
    'gc_per_match': 0.05,
    'positions_played': 0.05,
    'fouls_commiting': -0.07
}

In [431]:
midfielders_features = list(weights_midfielders.keys())
X_midfield = midfielders[midfielders_features].apply(pd.to_numeric, errors="coerce")

In [432]:
mu_midfield = X_midfield.mean()
sigma_midfield = X_midfield.std(ddof=0)
sigma_midfield = sigma_midfield.replace(0, 1e-9)
Z_midfield = (X_midfield - mu_midfield) / sigma_midfield
w_midfield = pd.Series(weights_midfielders)
midfielder_score = (Z_midfield * w_midfield).sum(axis=1)
midfielders['midfielder_score'] = midfielder_score
midfielders["rating"] = 50 + 10 * midfielder_score

In [433]:
midfielders_matches = df[df['position'] == 'M']['matches'].copy()

In [434]:
min_matches_midfield = midfielders_matches.min()
max_matches_midfield = midfielders_matches.max()
experience_factor_m = 0.85 + 0.15 * (midfielders_matches - min_matches_midfield) / (max_matches_midfield - min_matches_midfield)
midfielders['rating'] = midfielders['rating'] * experience_factor_m

In [435]:
midfielders = midfielders.sort_values(by='rating', ascending=False)

In [436]:
midfielders[['name','rating']].round(3)

,name,rating
23,Vule,57.724
18,Martin,57.533
14,Mario,53.982
6,Borjan,52.625
2,Simikj,52.505
16,Dare,49.676
3,Stef,47.312
25,Davide,45.849
39,Ron,42.867
31,Theo,42.536


In [437]:
midfielders[['name','rating']].round(3)

,name,rating
23,Vule,57.724
18,Martin,57.533
14,Mario,53.982
6,Borjan,52.625
2,Simikj,52.505
16,Dare,49.676
3,Stef,47.312
25,Davide,45.849
39,Ron,42.867
31,Theo,42.536


In [438]:
avg_midfielders_rating = round((midfielders['rating'].sum())/midfielders['rating'].size, 3)
print(f"Average Rating of Midfielder is {avg_midfielders_rating}")

Average Rating of Midfielder is 44.464


## Defenders

In [439]:
defenders = df[df['position'] =='D'].copy()

In [440]:
defenders

,name,position,secondary_position,matches,goals,assists,gc_per_match,hat-trick,hat-trick-assists,w_percentage,...,interceptions_overall_ability,fouls_commiting,stamina_rating,positions_played,leadership_measure,passing,physical,gk_ability,skill,positioning_sense
0,Daco,D,GK,36,5,11,0.44,0,0,0.36,...,5.50,4.25,5.00,5.00,3.25,4.00,5.75,9.50,3.00,4.50
1,Leo,D,GK,34,40,19,1.74,3,3,0.35,...,8.50,5.00,8.50,9.00,7.50,8.25,9.00,7.25,7.00,8.00
7,Ancho,D,NaN,27,2,0,0.07,0,0,0.37,...,4.00,5.75,5.25,2.00,2.75,2.00,3.00,1.50,2.00,3.75
9,Hristijan,D,NaN,25,14,21,1.40,0,1,0.52,...,6.50,3.50,4.00,4.75,4.75,6.25,8.00,3.00,5.50,7.00
10,Maki,D,GK,23,9,16,1.09,0,1,0.52,...,8.25,3.50,8.75,7.50,7.00,7.00,5.75,5.75,6.25,8.00
13,Bale,D,NaN,13,2,2,0.31,0,0,0.23,...,6.00,4.75,3.75,2.75,7.25,3.50,8.75,4.00,2.75,4.75
30,Gorjan,D,NaN,2,0,0,0.00,0,0,0.50,...,4.50,5.00,5.00,3.00,3.00,3.50,7.00,3.50,2.50,3.50
38,Stefan,D,NaN,1,1,0,1.00,0,0,1.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
41,Hris,D,NaN,1,0,0,0.00,0,0,0.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [441]:
weights_defenders = {
    'interceptions_overall_ability': 0.2,
    'positioning_sense': 0.15,
    'physical': 0.15,
    'stamina_rating': 0.13,
    'leadership_measure': 0.1,
    'passing': 0.1,
    'positions_played': 0.07,
    'goals': 0.07,
    'gc_per_match': 0.05,
    'skill': 0.04,
    'assists': 0.04,
    'fouls_commiting': -0.1
}

In [442]:
defenders_features = list(weights_defenders.keys())
X_defenders = defenders[defenders_features].apply(pd.to_numeric, errors="coerce")

In [443]:
mu_defenders = X_defenders.mean()
sigma_defenders = X_defenders.std(ddof=0)
sigma_defenders = sigma_defenders.replace(0, 1e-9)
Z_defenders = (X_defenders - mu_defenders) / sigma_defenders
w_defenders = pd.Series(weights_defenders)
defenders_score = (Z_defenders * w_defenders).sum(axis=1)
defenders['defenders_score'] = defenders_score
defenders['rating'] = 50 + 10 * defenders_score

In [444]:
defenders_matches = df[df['position'] == 'D']['matches'].copy()

In [445]:
min_matches_defenders = defenders_matches.min()
max_matches_defenders = defenders_matches.max()
experience_factor_d = 0.85 + 0.15 * (defenders_matches - min_matches_defenders) / (max_matches_defenders - min_matches_defenders)

defenders['rating'] = defenders['rating'] * experience_factor_d

In [446]:
defenders = defenders.sort_values(by='rating', ascending=False)

In [447]:
defenders[['name','rating']].round(3)

,name,rating
1,Leo,65.176
10,Maki,57.731
9,Hristijan,52.312
0,Daco,45.519
13,Bale,42.832
38,Stefan,42.071
41,Hris,41.331
30,Gorjan,35.208
7,Ancho,34.472


In [448]:
avg_defenders_rating = round((defenders['rating'].sum())/defenders['rating'].size, 3)
print(f'Average Defenders Rating is {avg_defenders_rating}')

Average Defenders Rating is 46.295


## Goalkeepers

In [449]:
goalkeepers = df[df['position'] =='GK'].copy()

In [450]:
goalkeepers

,name,position,secondary_position,matches,goals,assists,gc_per_match,hat-trick,hat-trick-assists,w_percentage,...,interceptions_overall_ability,fouls_commiting,stamina_rating,positions_played,leadership_measure,passing,physical,gk_ability,skill,positioning_sense
4,Neno,GK,NaN,29,0,6,0.21,0,0,0.38,...,3.0,8.50,5.00,2.75,1.00,3.25,5.75,8.00,2.50,2.25
15,Stec,GK,NaN,11,0,1,0.09,0,0,0.55,...,1.0,1.00,2.25,1.00,1.25,3.00,1.50,8.00,1.25,2.50
19,Fich,GK,NaN,8,2,1,0.38,0,0,0.25,...,5.0,3.25,6.00,5.25,3.00,5.25,6.25,9.00,4.25,4.25
21,Nuh,GK,NaN,4,0,1,0.25,0,0,0.50,...,1.5,1.00,2.50,1.00,3.25,3.00,2.25,7.25,1.50,2.00
27,Fishboi,GK,NaN,2,0,0,0.00,0,0,0.50,...,1.5,2.25,2.50,2.00,1.00,1.50,1.25,2.50,1.25,1.33
28,Silbo,GK,NaN,2,0,0,0.00,0,0,0.50,...,1.0,1.00,1.50,1.00,1.00,1.00,1.00,8.50,1.00,1.00
43,Ter Stegen,GK,NaN,1,0,0,0.00,0,0,0.00,...,5.0,3.50,7.00,3.50,3.00,5.00,8.00,8.00,5.50,5.50


In [451]:
goalkeepers[['name','gk_ability']].round(3)

,name,gk_ability
4,Neno,8.00
15,Stec,8.00
19,Fich,9.00
21,Nuh,7.25
27,Fishboi,2.50
28,Silbo,8.50
43,Ter Stegen,8.00


In [452]:
weights_goalkeepers = {
    'gk_ability': 0.65,
    'passing': 0.13, 
    'positioning_sense': 0.1,
    'leadership_measure': 0.07,
    'gc_per_match': 0.05
}

In [453]:
goalkeepers_features = list(weights_goalkeepers.keys())
X_goalkeepers = goalkeepers[goalkeepers_features].apply(pd.to_numeric, errors="coerce")

In [454]:
mu_goalkeepers = X_goalkeepers.mean()
sigma_goalkeepers = X_goalkeepers.std(ddof=0)
sigma_goalkeepers = sigma_goalkeepers.replace(0, 1e-9)
Z_goalkeepers = (X_goalkeepers - mu_goalkeepers) / sigma_goalkeepers
w_goalkeepers = pd.Series(weights_goalkeepers)
goalkeepers_score = (Z_goalkeepers * w_goalkeepers).sum(axis=1)
goalkeepers['goalkeepers_score'] = goalkeepers_score
goalkeepers['rating'] = 50 + 10 * goalkeepers_score

In [455]:
goalkeepers_matches = df[df['position'] == 'GK']['matches'].copy()

In [456]:
min_matches_goalkeepers = goalkeepers_matches.min()
max_matches_goalkeepers = goalkeepers_matches.max()
experience_factor_gk = 0.85 + 0.15 * (goalkeepers_matches - min_matches_goalkeepers) / (max_matches_goalkeepers - min_matches_goalkeepers)

goalkeepers['rating'] = goalkeepers['rating'] * experience_factor_gk

In [457]:
goalkeepers = goalkeepers.sort_values(by='rating', ascending=False)

In [458]:
goalkeepers[['name','rating']].round(3)

,name,rating
19,Fich,53.167
4,Neno,51.605
43,Ter Stegen,47.559
15,Stec,46.348
21,Nuh,43.759
28,Silbo,42.456
27,Fishboi,26.586


## Merging the ratings

In [459]:
df_with_ratings = pd.concat([goalkeepers,defenders,midfielders,attackers], ignore_index=True)

In [460]:
df_with_ratings

,name,position,secondary_position,matches,goals,assists,gc_per_match,hat-trick,hat-trick-assists,w_percentage,...,passing,physical,gk_ability,skill,positioning_sense,goalkeepers_score,rating,defenders_score,midfielder_score,attacker_score
0,Fich,GK,NaN,8,2,1,0.38,0,0,0.25,...,5.25,6.25,9.00,4.25,4.25,0.990595,53.166530,NaN,NaN,NaN
1,Neno,GK,NaN,29,0,6,0.21,0,0,0.38,...,3.25,5.75,8.00,2.50,2.25,0.160533,51.605332,NaN,NaN,NaN
2,Ter Stegen,GK,NaN,1,0,0,0.00,0,0,0.00,...,5.00,8.00,8.00,5.50,5.50,0.595142,47.558709,NaN,NaN,NaN
3,Stec,GK,NaN,11,0,1,0.09,0,0,0.55,...,3.00,1.50,8.00,1.25,2.50,0.129444,46.348189,NaN,NaN,NaN
4,Nuh,GK,NaN,4,0,1,0.25,0,0,0.50,...,3.00,2.25,7.25,1.50,2.00,0.052531,43.758526,NaN,NaN,NaN
5,Silbo,GK,NaN,2,0,0,0.00,0,0,0.50,...,1.00,1.00,8.50,1.00,1.00,-0.036450,42.456083,NaN,NaN,NaN
6,Fishboi,GK,NaN,2,0,0,0.00,0,0,0.50,...,1.50,1.25,2.50,1.25,1.33,-1.891795,26.586249,NaN,NaN,NaN
7,Leo,D,GK,34,40,19,1.74,3,3,0.35,...,8.25,9.00,7.25,7.00,8.00,NaN,65.176454,1.573994,NaN,NaN
8,Maki,D,GK,23,9,16,1.09,0,1,0.52,...,7.00,5.75,5.75,6.25,8.00,NaN,57.731113,1.113734,NaN,NaN
9,Hristijan,D,NaN,25,14,21,1.40,0,1,0.52,...,6.25,8.00,3.00,5.50,7.00,NaN,52.311705,0.489984,NaN,NaN


In [461]:
df_with_ratings.sort_values(by='rating',ascending=False)

,name,position,secondary_position,matches,goals,assists,gc_per_match,hat-trick,hat-trick-assists,w_percentage,...,passing,physical,gk_ability,skill,positioning_sense,goalkeepers_score,rating,defenders_score,midfielder_score,attacker_score
34,Hito,A,NaN,26,57,22,3.04,10,2,0.46,...,7.50,8.00,3.00,8.00,8.25,NaN,66.300775,NaN,NaN,1.668550
7,Leo,D,GK,34,40,19,1.74,3,3,0.35,...,8.25,9.00,7.25,7.00,8.00,NaN,65.176454,1.573994,NaN,NaN
35,Prem,A,NaN,27,38,19,2.11,7,0,0.41,...,7.75,6.75,3.25,8.50,8.00,NaN,58.347717,NaN,NaN,0.834772
8,Maki,D,GK,23,9,16,1.09,0,1,0.52,...,7.00,5.75,5.75,6.25,8.00,NaN,57.731113,1.113734,NaN,NaN
16,Vule,M,NaN,3,8,6,4.67,1,1,1.00,...,8.75,9.75,4.50,9.50,9.25,NaN,57.723887,NaN,1.719183,NaN
17,Martin,M,NaN,9,9,7,1.78,2,0,0.33,...,9.00,8.75,3.50,9.25,9.00,NaN,57.532749,NaN,1.490874,NaN
18,Mario,M,NaN,11,13,8,1.91,0,0,0.55,...,8.25,7.25,2.75,7.50,8.50,NaN,53.981670,NaN,1.028410,NaN
0,Fich,GK,NaN,8,2,1,0.38,0,0,0.25,...,5.25,6.25,9.00,4.25,4.25,0.990595,53.166530,NaN,NaN,NaN
19,Borjan,M,GK,27,31,19,1.85,4,1,0.63,...,5.25,8.25,4.75,5.50,7.00,NaN,52.624694,NaN,0.435414,NaN
20,Simikj,M,NaN,34,10,24,1.00,0,1,0.41,...,7.50,2.00,1.25,7.50,6.50,NaN,52.504628,NaN,0.250463,NaN


## Selected players

In [462]:
to_drop = ['name', 'position','secondary_position','hat-trick','hat-trick-assists','w_percentage','motm','potm','loss_percentage','defenders_score','midfielder_score','attacker_score','goalkeepers_score','fouls_commiting', 'matches']
attributes_list = [attribute for attribute in df_with_ratings.columns.tolist() if attribute not in to_drop]
attributes_list

['goals',
 'assists',
 'gc_per_match',
 'interceptions_overall_ability',
 'stamina_rating',
 'positions_played',
 'leadership_measure',
 'passing',
 'physical',
 'gk_ability',
 'skill',
 'positioning_sense',
 'rating']

I assigned these parameters with testing which is good which not ... but i get one conclusion for sure:
* The importance of the *goals*, *passing* and *stamina_rating* is the most important in small football from my knowledge.

In [463]:
#CONSTS
EPS = 1e-9
THRESHOLD_CONSTANT = 0.5  #because the normalization process is from 0 to 1, we need to somehow increase the values by a constant that will be valid for the range od [0.5,1]. This constant will scale it correctly.
THRESH = { #attribute-specific tolerances (percentage wise)
    'rating': 0.05,
    'skill': 0.07,
    'passing': 0.07,
    'physical': 0.07,
    'positioning_sense': 0.08,
    'stamina_rating': 0.07,
    'gk_ability': 0.1,
    'gc_per_match': 0.1,
    'goals': 0.15,
    'assists': 0.15,
    'leadership_measure': 0.12,
    'positions_played': 0.12,
    'interceptions_overall_ability': 0.1
}
THRESH = {k: v * THRESHOLD_CONSTANT for k, v in THRESH.items()}  #scaling for keeping it correct

WEIGHTS_OF_IMPORTANCE = { #attribute importance
#41
    'skill': 0.11,  # So important to be good in a small space
    'positioning_sense': 0.11, # The most important thing alongside skill, to know how to position for the ball and to move with the team
    'stamina_rating': 0.1,  # Constant movement and so much energy during the game
    'gc_per_match': 0.09,  # Goal contribution is important to score a goal

#25
    'goals': 0.09,  # Scoring impact
    'assists': 0.08,  # Creating opportunities
    'passing': 0.08,  # Passing is more important to bigger football, but here, with short passes easily can be done to make good plays 
#21
    'physical': 0.06,  # Needed attribute so the player can keep the ball while threatened from the opponent
    'interceptions_overall_ability': 0.06,  # Defensive contribution
    'leadership_measure': 0.05,  # Team organization (not so critical in hobby football, but sometimes important) 
    
#13
    'positions_played': 0.04,  # Flexibility is good, but not that critical in 6v6 football
    'gk_ability': 0.05,  # Relevant for key moments and stability on the goal
    'rating': 0.04,  # general rating must not be taken as something very important because sometimes some statistic are inputting in the rating so much, besides looking the fact that they are not the most important for the player to be considered as a key or not
}

MEAN_METRICS = ['rating', 'passing', 'physical', 'skill', 'positioning_sense', 'stamina_rating', 'leadership_measure',
                'gc_per_match', 'gk_ability', 'positions_played',
                'interceptions_overall_ability']  #these are some ability ratings that need to be averaged per team
SUM_METRICS = ['goals', 'assists']  #these are counting metrics that need to be summarized per team


def total_rating_per_team(team: pd.DataFrame):
    return sum(player['rating'] for player in team)


def attribute_team_statistics(team_df: pd.DataFrame) -> dict:
    stats = {}
    for attr in attributes_list:
        if attr in MEAN_METRICS:
            stats[attr] = float(team_df[attr].mean())
        if attr in SUM_METRICS:
            stats[attr] = float(team_df[attr].sum())
    return stats


def per_attribute_diff(team_a_summary_stats: dict, team_b_summary_stats: dict) -> dict:
    diffs = {}
    for attr in attributes_list:
        A, B = team_a_summary_stats[attr], team_b_summary_stats[attr]
        diffs[attr] = abs(A - B) / max(abs(A), abs(B), EPS)
    return diffs


def normalize_df(df: pd.DataFrame, columns: list) -> pd.DataFrame:  #normalizing the columns in range [0.5, 1]
    df_copy = df.copy()
    for column in columns:
        col = df[column]
        col_min, col_max = np.nanmin(col), np.nanmax(col)
        col_range = col_max - col_min + EPS
        df_copy[column] = 0.5 + 0.5 * (col - col_min) / col_range
    return df_copy

In [464]:
selected_players = ['Stec','Neno','Vule','Hito','Borjan','Prem','Leo','Hristijan','Simikj','Stef','Ancho','Maki']
#selected_players = ['Fishboi','Neno','Leo','Prem','Stef','Simikj','Ancho','Dzhoker','Hristijan','Grekos','Borjan','Daco']

In [465]:
match_players_df = df_with_ratings[df_with_ratings['name'].isin(selected_players)]
normalized_match_players_df = normalize_df(match_players_df, attributes_list)

In [466]:
normalized_match_players_df

,name,position,secondary_position,matches,goals,assists,gc_per_match,hat-trick,hat-trick-assists,w_percentage,...,passing,physical,gk_ability,skill,positioning_sense,goalkeepers_score,rating,defenders_score,midfielder_score,attacker_score
1,Neno,GK,NaN,29,0.500000,0.625000,0.515217,0,0,0.38,...,0.592593,0.757576,1.000000,0.575758,0.500000,0.160533,0.769146,NaN,NaN,NaN
3,Stec,GK,NaN,11,0.500000,0.520833,0.502174,0,0,0.55,...,0.574074,0.500000,1.000000,0.500000,0.517857,0.129444,0.686560,NaN,NaN,NaN
7,Leo,D,GK,34,0.850877,0.895833,0.681522,3,3,0.35,...,0.962963,0.954545,0.944444,0.848485,0.910714,NaN,0.982338,1.573994,NaN,NaN
8,Maki,D,GK,23,0.578947,0.833333,0.610870,0,1,0.52,...,0.870370,0.757576,0.833333,0.803030,0.910714,NaN,0.865377,1.113734,NaN,NaN
9,Hristijan,D,NaN,25,0.622807,0.937500,0.644565,0,1,0.52,...,0.814815,0.893939,0.629630,0.757576,0.839286,NaN,0.780242,0.489984,NaN,NaN
15,Ancho,D,NaN,27,0.517544,0.500000,0.500000,0,0,0.37,...,0.500000,0.590909,0.518519,0.545455,0.607143,NaN,0.500000,-1.414462,NaN,NaN
16,Vule,M,NaN,3,0.570175,0.625000,1.000000,1,1,1.00,...,1.000000,1.000000,0.740741,1.000000,1.000000,NaN,0.865264,NaN,1.719183,NaN
19,Borjan,M,GK,27,0.771930,0.895833,0.693478,4,1,0.63,...,0.740741,0.909091,0.759259,0.757576,0.839286,NaN,0.785159,NaN,0.435414,NaN
20,Simikj,M,NaN,34,0.587719,1.000000,0.601087,0,1,0.41,...,0.907407,0.530303,0.500000,0.878788,0.803571,NaN,0.783273,NaN,0.250463,NaN
22,Stef,M,GK,32,0.692982,0.750000,0.607609,2,0,0.44,...,0.666667,0.681818,0.814815,0.636364,0.678571,NaN,0.701707,NaN,-0.225351,NaN


In [467]:
match_gks = normalized_match_players_df[normalized_match_players_df['position'] == 'GK']

In [468]:
match_outfield = normalized_match_players_df[normalized_match_players_df['position'] != 'GK']

In [469]:
match_gks = match_gks.sort_values(by='rating', ascending=False).reset_index(drop=True)
match_outfield = match_outfield.sort_values(by='rating', ascending=False).reset_index(drop=True)

In [470]:
#Snake algorithm - used for the initial split of the teams
team_a = []
team_b = []
for i in range(0, len(match_outfield), 4):
    picks = match_outfield.iloc[i:i+4]
    if len(picks) >= 4:
        team_a += [picks.iloc[0], picks.iloc[3]]
        team_b += [picks.iloc[1], picks.iloc[2]]
    else:
        for j, row in picks.iterrows():
            if len(team_a) <= len(team_b):
                team_a.append(row)
            else:
                team_b.append(row)

In [471]:
team_a = pd.DataFrame(team_a)
team_b = pd.DataFrame(team_b)

In [472]:
team_a

,name,position,secondary_position,matches,goals,assists,gc_per_match,hat-trick,hat-trick-assists,w_percentage,...,passing,physical,gk_ability,skill,positioning_sense,goalkeepers_score,rating,defenders_score,midfielder_score,attacker_score
0,Hito,A,NaN,26,1.000000,0.958333,0.822826,10,2,0.46,...,0.907407,0.893939,0.629630,0.909091,0.928571,NaN,1.000000,NaN,NaN,1.66855
3,Maki,D,GK,23,0.578947,0.833333,0.610870,0,1,0.52,...,0.870370,0.757576,0.833333,0.803030,0.910714,NaN,0.865377,1.113734,NaN,NaN
4,Vule,M,NaN,3,0.570175,0.625000,1.000000,1,1,1.00,...,1.000000,1.000000,0.740741,1.000000,1.000000,NaN,0.865264,NaN,1.719183,NaN
7,Hristijan,D,NaN,25,0.622807,0.937500,0.644565,0,1,0.52,...,0.814815,0.893939,0.629630,0.757576,0.839286,NaN,0.780242,0.489984,NaN,NaN
8,Stef,M,GK,32,0.692982,0.750000,0.607609,2,0,0.44,...,0.666667,0.681818,0.814815,0.636364,0.678571,NaN,0.701707,NaN,-0.225351,NaN


In [473]:
team_b

,name,position,secondary_position,matches,goals,assists,gc_per_match,hat-trick,hat-trick-assists,w_percentage,...,passing,physical,gk_ability,skill,positioning_sense,goalkeepers_score,rating,defenders_score,midfielder_score,attacker_score
1,Leo,D,GK,34,0.850877,0.895833,0.681522,3,3,0.35,...,0.962963,0.954545,0.944444,0.848485,0.910714,NaN,0.982338,1.573994,NaN,NaN
2,Prem,A,NaN,27,0.833333,0.895833,0.721739,7,0,0.41,...,0.925926,0.818182,0.648148,0.939394,0.910714,NaN,0.875063,NaN,NaN,0.834772
5,Borjan,M,GK,27,0.771930,0.895833,0.693478,4,1,0.63,...,0.740741,0.909091,0.759259,0.757576,0.839286,NaN,0.785159,NaN,0.435414,NaN
6,Simikj,M,NaN,34,0.587719,1.000000,0.601087,0,1,0.41,...,0.907407,0.530303,0.500000,0.878788,0.803571,NaN,0.783273,NaN,0.250463,NaN
9,Ancho,D,NaN,27,0.517544,0.500000,0.500000,0,0,0.37,...,0.500000,0.590909,0.518519,0.545455,0.607143,NaN,0.500000,-1.414462,NaN,NaN


In [474]:
def calculate_imbalance_scores(team_a: pd.DataFrame, team_b: pd.DataFrame):
    #statistics for team A and B
    team_a_statistics = attribute_team_statistics(team_a)
    team_b_statistics = attribute_team_statistics(team_b)

    #difference between the statistics for team A and B
    differences_dictionary = per_attribute_diff(team_a_statistics, team_b_statistics)
    total_contribution = 0.0  #total imbalance score
    flag_dict = {}  #flags for every attribute

    for attr in attributes_list:
        threshold = THRESH.get(attr, 0.1)
        weight = WEIGHTS_OF_IMPORTANCE.get(attr, 0.1)
        difference = differences_dictionary[attr]
        contribution = weight * min(1, max(difference - threshold, 0) / threshold)
        total_contribution += contribution
        flag_dict[attr] = {
            "diff": difference,
            "threshold": threshold,
            "weight": weight,
            "ok": difference <= threshold,
            "contribution": contribution
        }
    # print('===================')
    # if total_contribution <= 0.3:
    #     print(f"balanced {total_contribution}")
    #     print('===================')
    #     print(f'{team_a['name']} vs {team_b['name']}')
    # elif total_contribution <= 0.5:
    #     print(f"moderate imbalance {total_contribution}")
    # else:
    #     print(f"high imbalance {total_contribution}")

    return total_contribution, flag_dict
    
    
def main_hill_climbing_algorithm(initial_team_a: pd.DataFrame,
                                 initial_team_b: pd.DataFrame,
                                 normalized_df: pd.DataFrame,
                                 target_ok_ratio=0.75,
                                 max_iterations=3):
    history_trace = []
    tA = initial_team_a.copy()
    tB = initial_team_b.copy()
    best_score, best_flags = calculate_imbalance_scores(tA, tB)
    history_trace.append((best_score, tA.copy()['name'].tolist(), tB.copy()['name'].tolist()))

    for iteration in range(max_iterations):
        improved_flag = False
        best_swap = None
        current_ok_ratio = sum(1 for v in best_flags.values() if v["ok"]) / len(best_flags)

        print(f'Iteration {iteration + 1} | Current OK ratio = {current_ok_ratio:.3f} | Target = {target_ok_ratio} |')

        if current_ok_ratio >= target_ok_ratio:
            print(f'Target OK ratio reached! Breaking at iteration {iteration + 1}')
            break
        for player_a in tA.itertuples():
            for player_b in tB.itertuples():
                temp_a = tA.drop(player_a.Index)._append(tB.loc[[player_b.Index]])
                temp_b = tB.drop(player_b.Index)._append(tA.loc[[player_a.Index]])

                temp_score, temp_flags = calculate_imbalance_scores(temp_a, temp_b)
                # print(f'TempScore is {temp_score}')
                new_ok_ratio = sum(1 for v in temp_flags.values() if v["ok"]) / len(temp_flags)
                best_ok_ratio = sum(1 for v in best_flags.values() if v["ok"]) / len(best_flags)

                if new_ok_ratio >= best_ok_ratio and temp_score < best_score:
                    best_score, best_flags = temp_score, temp_flags
                    best_swap = (temp_a, temp_b)
                    improved_flag = True
                    print('New Team Split Flags')
                    [print(f'{key}')for key, value in temp_flags.items() if value['ok']]
                    print('Best Team Split Flags')
                    [print(f'{key}')for key, value in best_flags.items() if value['ok']]
        if improved_flag:
            tA, tB = best_swap
            history_trace.append((best_score, tA.copy()['name'].tolist(), tB.copy()['name'].tolist()))

    return tA, tB, best_score, best_flags, history_trace

In [475]:
#TESTING
team_a_stats = attribute_team_statistics(team_a)
team_b_stats = attribute_team_statistics(team_b)
differences_a_b = per_attribute_diff(team_a_stats, team_b_stats)
score, _ = calculate_imbalance_scores(team_a, team_b)
final_team_a, final_team_b, best_score, best_flags, history_trace = main_hill_climbing_algorithm(team_a, team_b, normalized_match_players_df)

Iteration 1 | Current OK ratio = 0.231 | Target = 0.75 |
New Team Split Flags
assists
stamina_rating
rating
Best Team Split Flags
assists
stamina_rating
rating
New Team Split Flags
assists
interceptions_overall_ability
stamina_rating
positions_played
passing
positioning_sense
Best Team Split Flags
assists
interceptions_overall_ability
stamina_rating
positions_played
passing
positioning_sense
New Team Split Flags
assists
gc_per_match
interceptions_overall_ability
positions_played
leadership_measure
gk_ability
skill
positioning_sense
Best Team Split Flags
assists
gc_per_match
interceptions_overall_ability
positions_played
leadership_measure
gk_ability
skill
positioning_sense
New Team Split Flags
assists
interceptions_overall_ability
stamina_rating
positions_played
passing
gk_ability
skill
positioning_sense
Best Team Split Flags
assists
interceptions_overall_ability
stamina_rating
positions_played
passing
gk_ability
skill
positioning_sense
New Team Split Flags
gc_per_match
stamina_rating


In [476]:
print(f'Team A statistics: {team_a_stats}')
print('---------------------------------------')
print(f'Team B statistics: {team_b_stats}')
print(f'Initial Team A: {team_a['name'].tolist()}')
print(f'Initial Team B: {team_b['name'].tolist()}')
print('---------------------------------------')
print(f'Imbalances function output SCORE: {score}')
print('---------------------------------------')
print(f'Best Players to Swap are the following teams:')
print(f'Team A: {final_team_a['name'].tolist()}')
print(f'Team B: {final_team_b['name'].tolist()}')
print('---------------------------------------')
print(f'Best Score gotten for now is {best_score}')

print('-------------HISTORY TRACE------------')
[print(f'{iter+1}: {item}') for iter, item in zip(range(history_trace.__sizeof__()),history_trace)]

Team A statistics: {'goals': 3.4649122806848265, 'assists': 4.104166666599826, 'gc_per_match': 0.7371739129919187, 'interceptions_overall_ability': 0.919999999944, 'stamina_rating': 0.8571428570918366, 'positions_played': 0.8218749999597657, 'leadership_measure': 0.882758620636861, 'passing': 0.8518518517997258, 'physical': 0.8454545454126723, 'gk_ability': 0.7296296295956104, 'skill': 0.8212121211731865, 'positioning_sense': 0.8714285713755101, 'rating': 0.8425180538681453}
---------------------------------------
Team B statistics: {'goals': 3.5614035087533087, 'assists': 4.187499999929687, 'gc_per_match': 0.6395652173609642, 'interceptions_overall_ability': 0.8199999999573333, 'stamina_rating': 0.7642857142479592, 'positions_played': 0.7531249999683594, 'leadership_measure': 0.7655172413426873, 'passing': 0.8074074073618658, 'physical': 0.760606060574472, 'gk_ability': 0.6740740740482853, 'skill': 0.793939393903765, 'positioning_sense': 0.8142857142408164, 'rating': 0.785166643902634

[None, None]

In [477]:
match_gks

,name,position,secondary_position,matches,goals,assists,gc_per_match,hat-trick,hat-trick-assists,w_percentage,...,passing,physical,gk_ability,skill,positioning_sense,goalkeepers_score,rating,defenders_score,midfielder_score,attacker_score
0,Neno,GK,NaN,29,0.5,0.625000,0.515217,0,0,0.38,...,0.592593,0.757576,1.0,0.575758,0.500000,0.160533,0.769146,NaN,NaN,NaN
1,Stec,GK,NaN,11,0.5,0.520833,0.502174,0,0,0.55,...,0.574074,0.500000,1.0,0.500000,0.517857,0.129444,0.686560,NaN,NaN,NaN


In [478]:
rating_team_a = final_team_a['rating'].sum()
rating_team_b = final_team_b['rating'].sum()

In [479]:
if rating_team_a < rating_team_b:
    final_team_a = pd.concat([pd.DataFrame([match_gks.iloc[0]]), final_team_a], ignore_index=True)
    final_team_b = pd.concat([pd.DataFrame([match_gks.iloc[1]]), final_team_b], ignore_index=True)
else:
    final_team_b = pd.concat([pd.DataFrame([match_gks.iloc[0]]), final_team_b], ignore_index=True)
    final_team_a = pd.concat([pd.DataFrame([match_gks.iloc[1]]), final_team_a], ignore_index=True)

In [480]:
final_team_a

,name,position,secondary_position,matches,goals,assists,gc_per_match,hat-trick,hat-trick-assists,w_percentage,...,passing,physical,gk_ability,skill,positioning_sense,goalkeepers_score,rating,defenders_score,midfielder_score,attacker_score
0,Stec,GK,NaN,11,0.500000,0.520833,0.502174,0,0,0.55,...,0.574074,0.500000,1.000000,0.500000,0.517857,0.129444,0.686560,NaN,NaN,NaN
1,Hito,A,NaN,26,1.000000,0.958333,0.822826,10,2,0.46,...,0.907407,0.893939,0.629630,0.909091,0.928571,NaN,1.000000,NaN,NaN,1.668550
2,Maki,D,GK,23,0.578947,0.833333,0.610870,0,1,0.52,...,0.870370,0.757576,0.833333,0.803030,0.910714,NaN,0.865377,1.113734,NaN,NaN
3,Hristijan,D,NaN,25,0.622807,0.937500,0.644565,0,1,0.52,...,0.814815,0.893939,0.629630,0.757576,0.839286,NaN,0.780242,0.489984,NaN,NaN
4,Stef,M,GK,32,0.692982,0.750000,0.607609,2,0,0.44,...,0.666667,0.681818,0.814815,0.636364,0.678571,NaN,0.701707,NaN,-0.225351,NaN
5,Prem,A,NaN,27,0.833333,0.895833,0.721739,7,0,0.41,...,0.925926,0.818182,0.648148,0.939394,0.910714,NaN,0.875063,NaN,NaN,0.834772


In [481]:
final_team_b

,name,position,secondary_position,matches,goals,assists,gc_per_match,hat-trick,hat-trick-assists,w_percentage,...,passing,physical,gk_ability,skill,positioning_sense,goalkeepers_score,rating,defenders_score,midfielder_score,attacker_score
0,Neno,GK,NaN,29,0.500000,0.625000,0.515217,0,0,0.38,...,0.592593,0.757576,1.000000,0.575758,0.500000,0.160533,0.769146,NaN,NaN,NaN
1,Leo,D,GK,34,0.850877,0.895833,0.681522,3,3,0.35,...,0.962963,0.954545,0.944444,0.848485,0.910714,NaN,0.982338,1.573994,NaN,NaN
2,Borjan,M,GK,27,0.771930,0.895833,0.693478,4,1,0.63,...,0.740741,0.909091,0.759259,0.757576,0.839286,NaN,0.785159,NaN,0.435414,NaN
3,Simikj,M,NaN,34,0.587719,1.000000,0.601087,0,1,0.41,...,0.907407,0.530303,0.500000,0.878788,0.803571,NaN,0.783273,NaN,0.250463,NaN
4,Ancho,D,NaN,27,0.517544,0.500000,0.500000,0,0,0.37,...,0.500000,0.590909,0.518519,0.545455,0.607143,NaN,0.500000,-1.414462,NaN,NaN
5,Vule,M,NaN,3,0.570175,0.625000,1.000000,1,1,1.00,...,1.000000,1.000000,0.740741,1.000000,1.000000,NaN,0.865264,NaN,1.719183,NaN


In [482]:
print(f'Imbalance score of final teams is {calculate_imbalance_scores(final_team_a, final_team_b)}')

Imbalance score of final teams is (0.08885097034610966, {'goals': {'diff': 0.10165975103607891, 'threshold': 0.075, 'weight': 0.09, 'ok': False, 'contribution': 0.0319917012432947}, 'assists': {'diff': 0.07234042553006793, 'threshold': 0.075, 'weight': 0.08, 'ok': True, 'contribution': 0.0}, 'gc_per_match': {'diff': 0.020424836597969877, 'threshold': 0.05, 'weight': 0.09, 'ok': True, 'contribution': 0.0}, 'interceptions_overall_ability': {'diff': 0.026755852840661648, 'threshold': 0.05, 'weight': 0.06, 'ok': True, 'contribution': 0.0}, 'stamina_rating': {'diff': 0.033962264147867495, 'threshold': 0.035, 'weight': 0.1, 'ok': True, 'contribution': 0.0}, 'positions_played': {'diff': 0.003472222221932974, 'threshold': 0.06, 'weight': 0.04, 'ok': True, 'contribution': 0.0}, 'leadership_measure': {'diff': 0.07526881719782624, 'threshold': 0.06, 'weight': 0.05, 'ok': False, 'contribution': 0.012724014331521871}, 'passing': {'diff': 0.011673151749882791, 'threshold': 0.035, 'weight': 0.08, 'ok

In [483]:
[print(f'{key}: {value}\n') for key, value in best_flags.items()]

goals: {'diff': 0.11529411764570241, 'threshold': 0.075, 'weight': 0.09, 'ok': False, 'contribution': 0.048352941174842894}

assists: {'diff': 0.10476190475941048, 'threshold': 0.075, 'weight': 0.08, 'ok': False, 'contribution': 0.031746031743371185}

gc_per_match: {'diff': 0.01969981237965917, 'threshold': 0.05, 'weight': 0.09, 'ok': True, 'contribution': 0.0}

interceptions_overall_ability: {'diff': 0.05947955389892349, 'threshold': 0.05, 'weight': 0.06, 'ok': False, 'contribution': 0.011375464678708183}

stamina_rating: {'diff': 0.008771929823792044, 'threshold': 0.035, 'weight': 0.1, 'ok': True, 'contribution': 0.0}

positions_played: {'diff': 0.023529411762860303, 'threshold': 0.06, 'weight': 0.04, 'ok': True, 'contribution': 0.0}

leadership_measure: {'diff': 0.08032128513411076, 'threshold': 0.06, 'weight': 0.05, 'ok': False, 'contribution': 0.016934404278425633}

passing: {'diff': 0.017699115042681634, 'threshold': 0.035, 'weight': 0.08, 'ok': True, 'contribution': 0.0}

physic

[None, None, None, None, None, None, None, None, None, None, None, None, None]